In [1]:
import json
import tiktoken

#import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma

from dotenv.ipython import load_dotenv
import os

In [2]:
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [28]:
pdf_file = r"C:\Users\user\OneDrive\Bureau\retirieval augmented generation\pdfs\BI_Cours.pdf"

In [29]:
pdf_loader = PyPDFLoader(pdf_file)

In [30]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "./pdfs")

In [31]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [32]:
chunks = loader.load_and_split(text_splitter)

In [33]:
len(chunks)

29

In [34]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

In [35]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="cours_bi_V2",
    persist_directory="./store"
)

In [36]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [37]:
retrieved_chunks=retriever.invoke("Quelles sont les sources de données utilisées dans le cours de Business Intelligence ?")

In [38]:
print(retrieved_chunks)

[Document(metadata={'total_pages': 28, 'moddate': '2026-02-16T21:46:26+00:00', 'source': 'pdfs\\BI_Cours.pdf', 'creator': 'Microsoft® PowerPoint®\xa0LTSC', 'page': 13, 'creationdate': '2026-02-16T21:46:26+00:00', 'page_label': '14', 'title': 'SYSTEME D’INFORMATION DECISIONNEL', 'author': 'Fatima Kalna', 'producer': 'Microsoft® PowerPoint®\xa0LTSC'}, page_content='L’informatique décisionnelle (Business Intelligence BI)\nLa Business Intelligence (BI) désigne l’ensemble des méthodes, outils et technologies\nqui permettent de collecter, stocker, analyser et transformer les données d’une\norganisation en informations utiles afin d’aider à la prise de décision.\nElle s’appuie notamment sur :\n•des sources de données (bases de données, ERP, CRM, fichiers, etc.),\n•des entrepôts de données (Data Warehouse),\n•L’analyse multidimensionnelle OLAP\n•Des outils de restitution et de présentation (tableaux de bord, rapports, indicateurs de performance –\nKPI).\n👉 Objectif principal : fournir aux déci

In [39]:
print(len(retrieved_chunks))

5


In [40]:
prompt_template = """
Answer the following question based only on provided context
The context is about business intelligence course. 
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 {context}
</context>
<question>
 {question}
</question>
"""

In [41]:
user_input = "Quelles sont les sources de données utilisées dans le cours de Business Intelligence ?"

In [42]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [43]:
context_for_query

"L’informatique décisionnelle (Business Intelligence BI)\nLa Business Intelligence (BI) désigne l’ensemble des méthodes, outils et technologies\nqui permettent de collecter, stocker, analyser et transformer les données d’une\norganisation en informations utiles afin d’aider à la prise de décision.\nElle s’appuie notamment sur :\n•des sources de données (bases de données, ERP, CRM, fichiers, etc.),\n•des entrepôts de données (Data Warehouse),\n•L’analyse multidimensionnelle OLAP\n•Des outils de restitution et de présentation (tableaux de bord, rapports, indicateurs de performance –\nKPI).\n👉 Objectif principal : fournir aux décideurs une vision fiable, synthétique et\npertinente de l’activité de l’entreprise pour améliorer la performance, anticiper les\ntendances et réduire les risques.. Exemple 2\nL’entreprise dispose d’un ERP avec une base de données transactionnelle SQL :\nTables principales sont: \n▪ CLIENT\n▪ PRODUIT\n▪ COMMANDE\n▪ LIGNE_COMMANDE\n▪ PAIEMENT\n▪ STOCK\n❓ Question po

In [44]:
len(relevant_document_chunks)

5

In [45]:
prompt = prompt_template.format(context=context_for_query, question=user_input)

In [46]:
print(prompt)


Answer the following question based only on provided context
The context is about business intelligence course. 
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 L’informatique décisionnelle (Business Intelligence BI)
La Business Intelligence (BI) désigne l’ensemble des méthodes, outils et technologies
qui permettent de collecter, stocker, analyser et transformer les données d’une
organisation en informations utiles afin d’aider à la prise de décision.
Elle s’appuie notamment sur :
•des sources de données (bases de données, ERP, CRM, fichiers, etc.),
•des entrepôts de données (Data Warehouse),
•L’analyse multidimensionnelle OLAP
•Des outils de restitution et de présentation (tableaux de bord, rapports, indicateurs de performance –
KPI).
👉 Objectif principal : fournir aux décideurs une vision fiable, synthétique et
pertinente de l’activité de l’entreprise pour amélior

In [47]:
resp = llm.invoke(prompt)

In [48]:
from IPython.display import Markdown

In [49]:
print(display(Markdown(resp.content)))

Les sources de données utilisées dans le cours de Business Intelligence comprennent : 
- des bases de données,
- des ERP (Enterprise Resource Planning),
- des CRM (Customer Relationship Management),
- des fichiers, etc.

None


In [50]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [51]:
response = RAG("information sur les entrepôts de données dans le cours de Business Intelligence ?")
print(display(Markdown(response)))

Un entrepôt de données (Data Warehouse) est une collection de données orientées sujet, intégrées, non volatiles et historisées, organisées pour le support d'un processus d'aide à la décision. L’entreposage de données (Data Warehousing) est le processus de collecte, d’intégration, de stockage et d’organisation des données issues des systèmes opérationnels afin de faciliter l’analyse et la prise de décision. Les données dans un entrepôt peuvent être classées en quatre catégories : données détaillées, données agrégées, données historiques et métadonnées. Les données détaillées représentent les événements élémentaires et récents, tandis que les données agrégées sont des résultats de synthèse. Les données historiques permettent de conserver l’évolution des données dans le temps, et les métadonnées décrivent les règles de gestion et les significations des indicateurs.

None


In [52]:
user_input = "j'ai faim et je veux manger quelque chose"
output = RAG(user_input)
print(output)

JE NE SAIS PAS


In [53]:
response = RAG("connaissez-vous les sources de données utilisées dans le cours de Business Intelligence ?")
print(display(Markdown(response)))

Oui, les sources de données utilisées dans le cours de Business Intelligence comprennent : des bases de données, des ERP, des CRM, des fichiers, etc.

None


In [54]:
user_input ="son rôle dans la prise de décision stratégique en entreprise"

In [55]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [56]:
user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [57]:
answer = RAG(user_input)
print(display(Markdown(answer)))

La Business Intelligence (BI) joue un rôle crucial dans la prise de décision stratégique en entreprise en fournissant aux décideurs une vision fiable, synthétique et pertinente de l’activité de l’entreprise. Elle permet d'analyser et de transformer les données en informations utiles, ce qui aide à améliorer la performance, anticiper les tendances et réduire les risques. Grâce à des outils et méthodes variés, la BI aide à comprendre le marché, à anticiper les risques et opportunités, et à améliorer la performance et la rentabilité.

None


In [58]:
groundedness_rater_system_message="""
Vous êtes chargé d’évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d’IA pour générer la réponse, ainsi qu’une réponse générée par l’IA à la question.

Dans l’entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l’IA commencera par ###Answer.

Critères d’évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1 — La métrique n’est pas respectée du tout
2 — La métrique n’est respectée que dans une mesure limitée
3 — La métrique est respectée dans une bonne mesure
4 — La métrique est respectée en grande partie
5 — La métrique est entièrement respectée

Métrique :
La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions :

Écrivez d’abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d’évaluation et attribuer un score.
"""

In [59]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [60]:
def evaluate(system_message,user_message_template, question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response= model.invoke(prompt)
    return juge_response.content


In [61]:
resp=evaluate(groundedness_rater_system_message, user_message_template, user_input)

In [62]:
print(display(Markdown(resp)))

Pour évaluer la réponse selon la métrique, voici les étapes à suivre :

1. **Comprendre la question** : La question porte sur le rôle de la Business Intelligence (BI) dans la prise de décision stratégique en entreprise.

2. **Analyser le contexte** : Le contexte fournit des informations sur le processus décisionnel en entreprise, les objectifs de la BI, et les outils et méthodes utilisés par la BI pour aider à la prise de décision. Il mentionne également des exemples concrets de décisions stratégiques, comme le lancement de nouveaux produits, l'optimisation des stocks, et l'amélioration de la gestion des relations clients.

3. **Comparer la réponse avec le contexte** : La réponse doit être dérivée uniquement des informations présentées dans le contexte. Il faut vérifier si la réponse utilise les informations du contexte pour expliquer le rôle de la BI dans la prise de décision stratégique.

4. **Évaluer la pertinence de la réponse** : La réponse doit être pertinente par rapport à la question et doit refléter fidèlement les informations du contexte.

**Évaluation de la réponse :**

- La réponse explique que la BI joue un rôle crucial dans la prise de décision stratégique en entreprise, ce qui est en accord avec le contexte qui décrit la BI comme un outil pour fournir une vision fiable et pertinente de l'activité de l'entreprise.
- La réponse mentionne que la BI aide à améliorer la performance, anticiper les tendances et réduire les risques, ce qui est également en ligne avec le contexte.
- La réponse parle de l'analyse et de la transformation des données en informations utiles, ce qui est directement dérivé du contexte qui décrit la BI comme un ensemble de méthodes pour collecter, stocker, analyser et transformer les données.
- La réponse mentionne la compréhension du marché, l'anticipation des risques et opportunités, et l'amélioration de la rentabilité, qui sont des objectifs de la BI mentionnés dans le contexte.

**Score :** 5

La réponse respecte entièrement la métrique car elle est dérivée uniquement des informations présentées dans le contexte et répond de manière précise à la question posée.

None
